   # HW1: ETL-пайплайн курсов валют ЦБ РФ
   
   **Что делает ноутбук:** Извлекает актуальные курсы валют из REST API ЦБ, преобразует их в структурированный DataFrame, сохраняет снэпшот в MinIO, а затем загружает в PostgreSQL с гарантией идемпотентности.
   **Источник:** REST API `https://www.cbr-xml-daily.ru/daily_json.js`
   **Целевая таблица:** `stg.cbr_rates` в PostgreSQL.

In [1]:
import os
import requests
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values
import boto3
import io

In [2]:

from dotenv import load_dotenv

load_dotenv()

DB_CONFIG = {
    'host': 'localhost',
    'port': 5432,
    'dbname': os.getenv('POSTGRES_DB'),
    'user': os.getenv('POSTGRES_USER'),
    'password': os.getenv('POSTGRES_PASSWORD')
}

In [3]:
API_URL = "https://www.cbr-xml-daily.ru/daily_json.js"

try:
    response = requests.get(API_URL, timeout=10)
    response.raise_for_status()  # Обработка статусов возвратов
    data = response.json()
    print(f"Успешно получены данные о курсах валют от {data['Date'][:10]}")
except requests.exceptions.RequestException as e:
    print(f"Ошибка запроса к API ЦБ: {e}")
    data = None

Успешно получены данные о курсах валют от 2026-11-24


In [4]:
if data:
    record_date = data['Date'][:10] # Берем только дату YYYY-MM-DD
    rows = []
    for code, info in data['Valute'].items():
        rows.append({
            'record_date': record_date,
            'currency_code': code,
            'currency_name': info['Name'],
            'nominal': info['Nominal'],
            'value': info['Value']
        })
    df = pd.DataFrame(rows)
    print(f"Преобразовано в DataFrame. Количество валют: {len(df)}")
    display(df.head())
else:
    df = pd.DataFrame()
    print("Данные не получены")

Преобразовано в DataFrame. Количество валют: 54


,record_date,currency_code,currency_name,nominal,value
0,2026-11-24,AUD,Австралийский доллар,1,54.5304
1,2026-11-24,AZN,Азербайджанский манат,1,48.3228
2,2026-11-24,DZD,Алжирских динаров,100,59.9682
3,2026-11-24,GBP,Фунт стерлингов,1,107.6314
4,2026-11-24,AMD,Армянских драмов,100,22.7969


In [5]:
from io import StringIO

MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "http://localhost:9000")
s3 = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=os.getenv("MINIO_ROOT_USER"),
    aws_secret_access_key=os.getenv("MINIO_ROOT_PASSWORD"),
)

# Проверяем, существует ли бакет, и создаем его, если нет
try:
    s3.head_bucket(Bucket="landing-zone")
    print("[INFO] Бакет 'landing-zone' уже существует.")
except Exception:
    s3.create_bucket(Bucket="landing-zone")
    print("[INFO] Бакет 'landing-zone' успешно создан.")

# Загружаем CSV в бакет
buffer = StringIO()
df.to_csv(buffer, index=False)
s3.put_object(
    Bucket="landing-zone",
    Key="cbr/daily_rates.csv",
    Body=buffer.getvalue().encode("utf-8"),
)
print(f"Загружено {len(df)} строк в s3://landing-zone/cbr/daily_rates.csv")

[INFO] Бакет 'landing-zone' успешно создан.
Загружено 54 строк в s3://landing-zone/cbr/daily_rates.csv


In [6]:
obj = s3.get_object(Bucket="landing-zone", Key="cbr/daily_rates.csv")
df_raw = pd.read_csv(io.BytesIO(obj["Body"].read()), parse_dates=["record_date"])
df_raw["record_date"] = df_raw["record_date"].dt.date  # приводим к date

data_tuples = [tuple(x) for x in df_raw.to_numpy()]
conn = psycopg2.connect(**DB_CONFIG)
try:
    with conn, conn.cursor() as cur:
        execute_values(
            cur,
            """
            INSERT INTO stg.cbr_rates (record_date, currency_code, currency_name, nominal, value)
            VALUES %s
            ON CONFLICT (record_date, currency_code) DO NOTHING
            """,
            data_tuples,
        )
    print(f"Загружено {len(data_tuples)} записей из MinIO в stg.cbr_rates")
except Exception as e:
    print(f"{e}")
    raise
finally:
    conn.close()

Загружено 54 записей из MinIO в stg.cbr_rates


In [7]:
conn = psycopg2.connect(**DB_CONFIG)
check_df = pd.read_sql("""
    SELECT record_date, currency_code, currency_name, value 
    FROM stg.cbr_rates 
    WHERE currency_code IN ('USD', 'EUR') 
    ORDER BY currency_code;
""", conn)
conn.close()

display(check_df)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_16076\1114884048.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  check_df = pd.read_sql("""


,record_date,currency_code,currency_name,value
0,2026-11-21,EUR,Евро,90.3445
1,2026-11-24,EUR,Евро,90.2815
2,2026-11-21,USD,Доллар США,82.1911
3,2026-11-24,USD,Доллар США,82.1488
